# seege_ usage tutorial 
for synthetic EEG evaluation

#### 1. Imports and basic setup

In [ ]:
import warnings
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Install the package first (from the repository root): pip install -e .
from seege_ import (TimeFidelity, FrequencyFidelity, TimeFrequencyFidelity, ComplexityFidelity,
                    SpatialFidelity, Diversity, Privacy, evaluation_score)
from seege_.preprocessing import load_pickle

warnings.filterwarnings("ignore", category=UserWarning)

DATA_DIR = Path("..") / "data"   # this notebook is in examples/
FS = 512

#### Data format

Every seege_ function accepts real and synthetic data in one of two formats:

| Shape | Meaning |
|---|---|
| `(N, T)` | N single-channel segments of T samples |
| `(N, C, T)` | N segments with C channels of T samples |

- Real and synthetic sets must share C, T, sampling rate and preprocessing; N may differ.
- With C > 1, the time, frequency, time-frequency and complexity metrics treat every channel of every segment as one signal; spatial fidelity requires C ≥ 2; diversity and privacy use one vector per segment.
- A single segment is passed as `x[None, :]` (shape `(1, T)`), or `x[None]` for multichannel data.
- Pairwise metrics (coherence, scalogram similarity, DCCA/MFDCCA) compare all pairs of signals, so their cost grows with the square of the number of signals: use `max_signals` (random subset, fixed seed) on large datasets.

#### 2. Data preprocessing

1. Load data
2. Downsampling
3. Normalization

#### 2.1. Load data

In [ ]:
real_data = load_pickle(DATA_DIR / "mayo_epileptic_1000seg.pkl")
synthetic_data = load_pickle(DATA_DIR / "synthetic_mayo_epileptic_1000seg.pkl")

# Print data shape
print(np.shape(real_data))       # (1000, 1536): 1000 single-channel segments of 3 s at 512 Hz
print(np.shape(synthetic_data))

# One example segment of each set
real_seg = real_data[4]          # shape (1536,)
synth_seg = synthetic_data[4]    # shape (1536,)

t = np.arange(len(real_seg)) / FS  # Time vector in seconds
plt.figure(figsize=(6, 3), facecolor=plt.rcParams['figure.facecolor'])

ax = sns.lineplot(
    x=t,
    y=real_seg,
    color="limegreen",
    label="Real Data"
)

sns.lineplot(
    x=t,
    y=synth_seg,
    color="lightskyblue",
    label="Synthetic Data",
    ax=ax
)

ax.set_title("Real and synthetic EEG segments")
ax.set_xlabel("Time (s)")
ax.set_ylabel("Amplitude")
ax.legend()
plt.tight_layout()
plt.show()


#### 3. Fidelity evaluation


#### 3.1. Time fidelity

In [ ]:
# Initialize class
sim = TimeFidelity()

# Compute Hjorth metrics
hjorth_dataset = sim.compute_hjorth_metrics(real_data, synthetic_data, verbose=True)

# Plot Hjorth metrics
hjorth_figures = sim.plot_hjorth_metrics(real_data, synthetic_data)

# Compute time fidelity score
evaluation_score.compute_time_fidelity_score(real_data, synthetic_data)

#### 3.2. Frequency fidelity

In [ ]:
# Initialize class
frequency_analysis = FrequencyFidelity(fs=FS)
# Compute frequency similarity metrics
# At dataset level
frequency_analysis.compare_relative_power(real_data, synthetic_data)
frequency_analysis.spectral_coherence(real_data, synthetic_data, max_signals=200)   # pairwise: subset
# At sample level: one segment as (1, T)
#frequency_analysis.compare_relative_power(real_data[:1], synthetic_data[:1])
#frequency_analysis.spectral_coherence(real_data[:1], synthetic_data[:1])

#Plot average PSD
# At dataset level
frequency_analysis.plot_psd(real_data, synthetic_data, scale="linear")
# At sample level
#frequency_analysis.plot_psd(real_data[5:6], synthetic_data[5:6], scale="log")

# Compute frequency fidelity score
# At dataset level
evaluation_score.compute_frequency_fidelity_score(real_data, synthetic_data, fs=FS, max_signals=200)
# At sample level
#evaluation_score.compute_frequency_fidelity_score(real_data[:1], synthetic_data[:1], fs=FS)

#### 3.3. Time-frequency fidelity

In [ ]:
# Initialize class
time_frequency_analysis = TimeFrequencyFidelity(fs=FS)

# Plot scalograms
# At sample level
time_frequency_analysis.plot_scalograms(real_data, synthetic_data, signal_index_real=0, signal_index_synth=0)
# At dataset level
time_frequency_analysis.plot_mean_scalograms(real_data, synthetic_data, freq_scale="log")

# Compute time-frequency similarity metrics
# At dataset level (pairwise: random subset of 100 signals per set)
time_frequency_analysis.compute_scalogram_similarity_metrics(real_data, synthetic_data, max_signals=100)
# At sample level
#time_frequency_analysis.compute_scalogram_similarity_metrics(real_data[:1], synthetic_data[:1])

# Compute burst statistics within the beta band
burst_results = time_frequency_analysis.compute_burst_statistics(real_data[:100], synthetic_data[:100], band=(13, 30))

# Compute time-frequency fidelity score
#evaluation_score.compute_time_frequency_fidelity_score(real_data[1:2], synthetic_data[3:4], fs=FS)
evaluation_score.compute_time_frequency_fidelity_score(real_data, synthetic_data, fs=FS, mode="all_vs_all",
                                                       max_signals=100)

#### 3.4. Complexity fidelity

In [ ]:
# Complexity metrics compare all pairs of signals and are slow: use a small random subset
# Initialize class for MFDCCA method (for DCCA and MFDFA, specify method accordingly)
cf = ComplexityFidelity(real_data, synthetic_data, method='MFDCCA', max_signals=10)

# Compute MFDCCA metrics
cf.compute_fractal_metrics()

# Plot MFDCCA metrics
cf.plot_metrics()

# Compute entropy metrics
entropy_metrics = cf.compute_entropy_complexity_metrics(real_data, synthetic_data, max_signals=50)

# Compute complexity fidelity score
evaluation_score.compute_complexity_fidelity_score(real_data, synthetic_data, max_signals=10)

#### 3.5. Spatial fidelity

Spatial fidelity compares the inter-channel correlation structure, so it requires multichannel data `(N, C, T)` with C ≥ 2. The example dataset is single-channel, so this cell only runs on multichannel data.

In [ ]:
if np.ndim(real_data) == 3 and np.shape(real_data)[1] >= 2:
    sf = SpatialFidelity()
    sf.evaluate(real_data, synthetic_data)
    evaluation_score.compute_spatial_fidelity_score(real_data, synthetic_data)
else:
    print("Spatial fidelity skipped: the data are single-channel (N, T).")

#### 4. Diversity evaluation




In [ ]:
# Initialize class (diversity is computed on band-power, Hjorth and inter-channel
# correlation features, which need the sampling rate)
div = Diversity(fs=FS)

# Real-vs-real baseline: the values a perfect generator could reach at this sample size.
# With subject IDs, split by subject: div.sanity_check(real_data, groups=subject_ids)
#base = div.sanity_check(real_data)

# Compute coverage diversity metrics
cov = div.compute_coverage_diversity(real_data, synthetic_data)

# Compute geometric diversity metrics
geom = div.compute_geometric_diversity(real_data, synthetic_data)

# Plot PCA and UMAP results
div.plot_embeddings("PCA", geom)
#div.plot_embeddings("UMAP", geom)

# Compute intrinsic diversity metrics
intr = div.compute_intrinsic_diversity(real_data, synthetic_data)

# Compute diversity score
evaluation_score.compute_diversity_score(real_data, synthetic_data, fs=FS)

#### 5. Privacy evaluation

In [ ]:
# Initialize class
privacy_analysis = Privacy()

# Compute nearest-neighbour distance metrics (DTW is O(T^2) per pair: use a subset)
distance_metrics = privacy_analysis.compute_distance_metrics(real_data[:100], synthetic_data[:100])
# Membership inference needs the task labels of the real segments:
#mir = privacy_analysis.compute_mir_metrics(real_data[:100], synthetic_data[:100], y_real)

# Compute privacy score
evaluation_score.compute_privacy_score(real_data[:100], synthetic_data[:100])